# SciGenX — Agentic RAG Scientific Research Assistant

This notebook builds an MVP that:
- reads research paper PDFs
- extracts page-wise text
- chunks text and creates embeddings
- stores vectors in FAISS
- retrieves evidence for a research question
- uses local Ollama (`llama3.2:latest`) for structured paper analysis
- compares papers and identifies potential research gaps
- generates a testable hypothesis and experiment plan
- produces a Markdown research report

> **Important:** A research gap is presented as a *potential gap based on the analyzed papers*, not as a guaranteed novel discovery.


## Step 0 — Environment check

In [3]:
import sys
from pathlib import Path

print("Python:", sys.version)
print("Working directory:", Path.cwd())


Python: 3.11.1 (tags/v3.11.1:a7a450f, Dec  6 2022, 19:58:39) [MSC v.1934 64 bit (AMD64)]
Working directory: c:\Users\siris\OneDrive\Desktop\SciGenX


## Step 1 — Imports

In [4]:
import os
import re
import json
import textwrap
from pathlib import Path

import fitz  # PyMuPDF
import numpy as np
import faiss
import ollama

from sentence_transformers import SentenceTransformer

print("Imports successful.")


c:\Users\siris\OneDrive\Desktop\SciGenX\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports successful.


## Step 2 — Project folders

In [5]:
PROJECT_ROOT = Path.cwd()

PAPERS_DIR = PROJECT_ROOT / "papers"
DATA_DIR = PROJECT_ROOT / "data"
SRC_DIR = PROJECT_ROOT / "src"
REPORTS_DIR = PROJECT_ROOT / "reports"

for folder in [PAPERS_DIR, DATA_DIR, SRC_DIR, REPORTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Paper folder:", PAPERS_DIR)


Project root: c:\Users\siris\OneDrive\Desktop\SciGenX
Paper folder: c:\Users\siris\OneDrive\Desktop\SciGenX\papers


## Step 3 — Add research papers

Place one or more `.pdf` research papers inside:

`SciGenX/papers/`

Then run the next cell.


In [6]:
pdf_files = sorted(PAPERS_DIR.glob("*.pdf"))

print(f"Found {len(pdf_files)} PDF file(s).")
for pdf in pdf_files:
    print("-", pdf.name)


Found 3 PDF file(s).
- compherencive_review.pdf
- plant_desease.pdf
- rice_desease.pdf


## Step 4 — Extract page-wise text from PDFs

In [7]:
def extract_pdf_pages(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []

    for page_number, page in enumerate(doc, start=1):
        text = page.get_text("text").strip()
        if text:
            pages.append({
                "paper": pdf_path.name,
                "page": page_number,
                "text": text
            })

    doc.close()
    return pages


all_pages = []

for pdf in pdf_files:
    pages = extract_pdf_pages(pdf)
    all_pages.extend(pages)
    print(f"{pdf.name}: {len(pages)} text pages")

print("Total extracted pages:", len(all_pages))


compherencive_review.pdf: 23 text pages
plant_desease.pdf: 47 text pages
rice_desease.pdf: 10 text pages
Total extracted pages: 80


## Step 5 — Clean text

In [8]:
def clean_text(text):
    text = re.sub(r"\s+", " ", text)
    return text.strip()


for item in all_pages:
    item["text"] = clean_text(item["text"])

print(all_pages[0]["text"][:1000] if all_pages else "No PDF text found.")


A comprehensive review on Plant Leaf Disease detection using Deep learning Sumaya Mustofa Department of Computer Science and Engineering Daffodil International University, Dhaka, Bangladesh sumaya15-3445@diu.edu.bd Md Mehedi Hasan Munna Department of Computer Science and Engineering Daffodil International University, Dhaka, Bangladesh mehedi15-3037@diu.edu.bd Yousuf Rayhan Emon Department of Computer Science and Engineering Daffodil International University, Dhaka, Bangladesh yousuf15-3220@diu.edu.bd Golam Rabbany Lecturer Department of CSE Daffodil International Univarsity Dhaka, Bangladesh rabbany.cse@diu.edu.bd Dr. Md Taimur Ahad Associate Professor Department of CSE Daffodil International Univarsity Dhaka, Bangladesh taimur.cse0396.c@diu.edu.bd


## Step 6 — Chunk the papers

In [9]:
def chunk_text(text, chunk_size=900, overlap=150):
    words = text.split()
    chunks = []

    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk = " ".join(words[start:end])

        if chunk.strip():
            chunks.append(chunk)

        if end == len(words):
            break

        start = end - overlap

    return chunks


chunks = []

for page in all_pages:
    page_chunks = chunk_text(page["text"])

    for chunk_id, chunk in enumerate(page_chunks):
        chunks.append({
            "paper": page["paper"],
            "page": page["page"],
            "chunk_id": chunk_id,
            "text": chunk
        })

print("Total chunks:", len(chunks))


Total chunks: 80


## Step 7 — Load embedding model

In [10]:
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Embedding model loaded:", EMBEDDING_MODEL_NAME)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1964.69it/s]


Embedding model loaded: all-MiniLM-L6-v2


## Step 8 — Create embeddings and FAISS index

In [11]:
print("PDF files:", len(pdf_files))
print("Extracted pages:", len(all_pages))
print("Chunks:", len(chunks))

if all_pages:
    print("\nFirst page text:")
    print(all_pages[0]["text"][:500])
else:
    print("\nNo page text was extracted from the PDF.")

PDF files: 3
Extracted pages: 80
Chunks: 80

First page text:
A comprehensive review on Plant Leaf Disease detection using Deep learning Sumaya Mustofa Department of Computer Science and Engineering Daffodil International University, Dhaka, Bangladesh sumaya15-3445@diu.edu.bd Md Mehedi Hasan Munna Department of Computer Science and Engineering Daffodil International University, Dhaka, Bangladesh mehedi15-3037@diu.edu.bd Yousuf Rayhan Emon Department of Computer Science and Engineering Daffodil International University, Dhaka, Bangladesh yousuf15-3220@diu.e


In [12]:
chunk_texts = [item["text"] for item in chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("Embedding shape:", embeddings.shape)
print("FAISS vectors:", index.ntotal)


Batches: 100%|██████████| 3/3 [00:03<00:00,  1.26s/it]

Embedding shape: (80, 384)
FAISS vectors: 80


## Step 9 — Save the vector index

In [13]:
INDEX_PATH = DATA_DIR / "scigenx.index"
CHUNKS_PATH = DATA_DIR / "chunks.json"

faiss.write_index(index, str(INDEX_PATH))

with open(CHUNKS_PATH, "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print("Saved:", INDEX_PATH)
print("Saved:", CHUNKS_PATH)


Saved: c:\Users\siris\OneDrive\Desktop\SciGenX\data\scigenx.index
Saved: c:\Users\siris\OneDrive\Desktop\SciGenX\data\chunks.json


## Step 10 — Semantic retrieval

In [14]:
def retrieve(query, top_k=5):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, ids = index.search(query_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], ids[0]):
        if idx == -1:
            continue

        item = chunks[int(idx)].copy()
        item["score"] = float(score)
        results.append(item)

    return results


query = "What methods are commonly used for detecting crop diseases?"

results = retrieve(query, top_k=5)

for i, result in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print("Paper:", result["paper"])
    print("Page:", result["page"])
    print("Score:", round(result["score"], 4))
    print(result["text"][:500])



--- Result 1 ---
Paper: plant_desease.pdf
Page: 31
Score: 0.5908
Plant Diseases recognition on images using Convolutional Neural Networks: A Systematic Review A PREPRINT Table 4 continued from previous page ID Study DataSet Name Type Crop Disease Name Pathogen Photinia serrulata Lindl N/A N/A Pinus massoniana Lamb N/A N/A Withered dicranopteris dichotoma N/A N/A Withered Mosla chinensis Maxim N/A N/A Miscanthus sinensis Anderss N/A N/A Acacia confusa Merr N/A N/A S58 Custom Tomato Tomato Leaf Miner Plague Phoma Rot Fungi Target Leaf Spot Fungi S64 Custom PDDS

--- Result 2 ---
Paper: plant_desease.pdf
Page: 23
Score: 0.5735
Plant Diseases recognition on images using Convolutional Neural Networks: A Systematic Review A PREPRINT B Summarization Table 4: Diseases and their Pathogens grouped by Dataset and Studies ID Study DataSet Name Type Crop Disease Name Pathogen S3, S4, S9, S11, S13, S14, S18, S21, S23, S26, S27, S28, S29, S35, S41, S48, S51, S53, S55, S56, S57, S60, S61, S62, S63, S

## Step 11 — Test Ollama

In [15]:
OLLAMA_MODEL = "llama3.2:latest"

response = ollama.chat(
    model=OLLAMA_MODEL,
    messages=[
        {
            "role": "user",
            "content": "Explain a research gap in 2 simple sentences."
        }
    ]
)

print(response["message"]["content"])


A research gap is a knowledge gap that prevents us from answering a specific research question or addressing a particular issue with sufficient evidence or data. It is a void in current research that needs to be filled by further investigation to advance our understanding of the topic and inform decision-making or policy development.


## Step 12 — Build a RAG context

In [16]:
def build_context(results):
    context_parts = []

    for i, item in enumerate(results, start=1):
        context_parts.append(
            f"[Evidence {i}] "
            f"Paper: {item['paper']} | Page: {item['page']} | "
            f"Similarity: {item['score']:.4f}\n"
            f"{item['text']}"
        )

    return "\n\n".join(context_parts)


context = build_context(results)
print(context[:4000])


[Evidence 1] Paper: plant_desease.pdf | Page: 31 | Similarity: 0.5908
Plant Diseases recognition on images using Convolutional Neural Networks: A Systematic Review A PREPRINT Table 4 continued from previous page ID Study DataSet Name Type Crop Disease Name Pathogen Photinia serrulata Lindl N/A N/A Pinus massoniana Lamb N/A N/A Withered dicranopteris dichotoma N/A N/A Withered Mosla chinensis Maxim N/A N/A Miscanthus sinensis Anderss N/A N/A Acacia confusa Merr N/A N/A S58 Custom Tomato Tomato Leaf Miner Plague Phoma Rot Fungi Target Leaf Spot Fungi S64 Custom PDDS Apple Scab Fungi Black Rot Fungi Ceder Rust Fungi Corn Cercospora leaf spot Fungi Common rust Fungi Northen leaf blight Fungi Tomato Early Blight Fungi Leaf Mold Fungi Septoria leaf spot Fungi Target Leaf Spot Fungi S65 Saiama Research Center Cucumbers N/A N/A S69 Custom Mango Bacterial Canker Bacterial Mildew Mango Fungi Phoma Blight Fungi Red Rust Alga Groundnut Leaf Spot Fungi Bud Necrosis Virus S70 Custom Pepper Tomato sp

## Step 13 — Ask the local LLM using retrieved evidence

In [17]:
def ask_rag(question, top_k=5):
    retrieved = retrieve(question, top_k=top_k)
    context = build_context(retrieved)

    prompt = f'''
You are SciGenX, a scientific research assistant.

Answer the research question using ONLY the retrieved evidence below.

Research question:
{question}

Retrieved evidence:
{context}

Instructions:
1. Give a concise evidence-based answer.
2. Mention paper names and page numbers where useful.
3. Do not invent facts.
4. If the evidence is insufficient, say so.
'''

    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[{"role": "user", "content": prompt}]
    )

    return {
        "answer": response["message"]["content"],
        "evidence": retrieved
    }


rag_result = ask_rag(
    "What are the main methods used in these papers?"
)

print(rag_result["answer"])


Based on the retrieved evidence, the main methods used in the papers are:

1. Convolutional Neural Networks (CNNs): 121 studies used CNNs for plant disease recognition, with a predominance of TensorFlow as the framework used (Evidence 5, Page 8).
2. Transfer Learning: 25 studies proposed new architectures, but 15 investigations customized classic algorithms to solve the problem of identifying plant diseases using CNNs (Evidence 5, Page 8).
3. Image Segmentation: 7 groups of methods were categorized, including Image Segmentation, as a practical approach (Evidence 3, Page 7).
4. Data Augmentation: 66 studies used customized data sets, with 45 studies using the PlantVillage data set (Evidence 3, Page 7).
5. Unsupervised Learning: 7 groups of methods were categorized, including Unsupervised Learning, as a practical approach (Evidence 3, Page 7).

Note that the evidence is sufficient to provide these concise answers, but it does not provide a comprehensive overview of the methods used in th

## Step 14 — Structured paper analysis

In [20]:
def analyze_paper(paper_name):
    paper_chunks = [c for c in chunks if c["paper"] == paper_name]

    if not paper_chunks:
        return {"error": f"Paper not found: {paper_name}"}

    # Only 5 chunks to keep Ollama fast
    selected = paper_chunks[:5]

    paper_context = "\n\n".join(
        f"[Page {c['page']}] {c['text'][:1200]}"
        for c in selected
    )

    prompt = f"""
Analyze this research paper using ONLY the text below.

Paper: {paper_name}

Text:
{paper_context}

Give a concise answer with these headings:

Problem:
Method:
Dataset:
Evaluation:
Findings:
Limitations:
Future Work:

Do not invent information.
"""

    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
        options={
            "num_predict": 500
        }
    )

    return response["message"]["content"]


paper_analyses = {}

for paper in sorted({c["paper"] for c in chunks}):
    print("Analyzing:", paper)
    paper_analyses[paper] = analyze_paper(paper)

print("\nAnalysis completed.")
print(json.dumps(paper_analyses, indent=2, ensure_ascii=False))

Analyzing: compherencive_review.pdf
Analyzing: plant_desease.pdf
Analyzing: rice_desease.pdf

Analysis completed.
{
  "compherencive_review.pdf": "Here is the concise analysis:\n\n**Problem:**\nLeaf disease is a common fatal disease for plants, and early diagnosis and detection are necessary to improve prognosis.\n\n**Method:**\nThis paper provides a systematic review of literature on leaf disease-based models using deep learning, including various models such as Vision Transformer, Deep convolutional neural network, Convolutional neural network, Residual Skip Network-based Super-Resolution for Leaf Disease Detection, Disease Detection Network, and YOLO.\n\n**Dataset:**\nPublicly available datasets are used in existing studies related to leaf disease detection.\n\n**Evaluation:**\nDifferent metrics such as accuracy, precision, recall, etc. are used to compare the performance of models.\n\n**Findings:**\nThe review highlights the advantages and limitations of different deep learning mod

## Step 15 — Compare papers

In [19]:
def compare_papers(analyses):
    compact = json.dumps(analyses, indent=2, ensure_ascii=False)

    prompt = f'''
You are comparing scientific papers.

Paper analyses:
{compact}

Identify:
1. Common approaches
2. Differences in methods
3. Dataset differences
4. Evaluation differences
5. Limitations shared across papers
6. Conflicting findings, if any
7. Potential research gaps

Important:
- Only use information present in the analyses.
- A gap must be described as a potential gap, not guaranteed novelty.
- Clearly distinguish evidence from inference.

Return a structured, readable answer.
'''

    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[{"role": "user", "content": prompt}]
    )

    return response["message"]["content"]


comparison = compare_papers(paper_analyses)
print(comparison)


Based on the provided analysis, I have identified the following key points:

**1. Common approaches**

* Both papers utilize deep learning techniques for plant leaf disease recognition.
* Both papers employ convolutional neural networks (CNN) as the primary method.

**2. Differences in methods**

* Paper 1 uses Vision Transformers (ViT), while Paper 2 uses Convolutional Neural Networks (CNN).
* Paper 1 mentions the exploration of using reinforcement learning methods, while Paper 2 does not.

**3. Dataset differences**

* Paper 1 uses publicly available free datasets, while Paper 2 uses multiple private datasets (PlantVillage Subset, PantVillage, Pest Image DataSet, TRES Taiwan Tea Research, UAV, Middle European Wood).
* Paper 1 mentions the difficulty in obtaining large annotated images, while Paper 2 does not provide information on dataset size.

**4. Evaluation differences**

* Paper 1 evaluates the improvement of Vision Transformer accuracy by 1% compared to CNN, while Paper 2 only 

## Step 16 — Generate a testable hypothesis

In [21]:
def generate_hypothesis(comparison_text):
    prompt = f'''
Based on the following literature comparison:

{comparison_text}

Generate one cautious, testable research hypothesis.

Return:
- Hypothesis
- Independent variable
- Dependent variable
- Baseline
- Expected outcome
- How it could be tested

Do not claim guaranteed novelty.
'''

    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[{"role": "user", "content": prompt}]
    )

    return response["message"]["content"]


hypothesis = generate_hypothesis(comparison)
print(hypothesis)


Based on the literature comparison, here is a cautious, testable research hypothesis:

**Research Hypothesis:**

"Deep learning architectures that combine reinforcement learning methods with convolutional neural networks (CNN) may improve the accuracy of plant leaf disease recognition systems compared to traditional CNN-based approaches, particularly when training on small, annotated datasets."

**Independent Variable:**

* Independent variable: Presence/absence of reinforcement learning methods in the deep learning architecture (i.e., whether the CNN is modified to incorporate reinforcement learning)

**Dependent Variable:**

* Dependent variable: Accuracy of plant leaf disease recognition systems

**Baseline:**

* Baseline: Traditional CNN-based approach without reinforcement learning methods (i.e., the current state-of-the-art method for plant leaf disease recognition)

**Expected Outcome:**

* Expected outcome: Improved accuracy of plant leaf disease recognition systems when using 

## Step 17 — Generate an experiment plan

In [22]:
def generate_experiment_plan(hypothesis_text):
    prompt = f'''
Create a practical experiment plan for this hypothesis:

{hypothesis_text}

Include:
1. Dataset
2. Preprocessing
3. Baseline model
4. Proposed method
5. Train/validation/test split
6. Evaluation metrics
7. Ablation or comparison
8. Expected risks
9. Reproducibility checklist

Keep it suitable for an engineering student research project.
'''

    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[{"role": "user", "content": prompt}]
    )

    return response["message"]["content"]


experiment_plan = generate_experiment_plan(hypothesis)
print(experiment_plan)


**Experiment Plan:**

**Title:** Investigating the Effect of Reinforcement Learning on Convolutional Neural Networks for Plant Leaf Disease Recognition

**Dataset:**

* Collect a dataset of 1000-2000 images of plant leaves with 5-10 classes of diseases (e.g., rust, powdery mildew, leaf spot, etc.)
* Use a combination of image sources, such as:
	+ Online datasets (e.g., PlantVillage, LeafIdentify)
	+ Laboratory-collected images
	+ Image segmentation of existing datasets
* Ensure the dataset is small and annotated, as per the research hypothesis

**Preprocessing:**

* Resize images to 224x224 pixels
* Apply random horizontal flipping, rotation, and color jittering to augment the dataset
* Normalize pixel values to the range [0, 1]
* Use data augmentation techniques to increase dataset size without increasing the number of original images

**Baseline Model:**

* Implement a traditional CNN-based approach using a pre-trained architecture (e.g., VGG16, ResNet50)
* Train the baseline model u

## Step 18 — Generate evidence-backed research report

In [23]:
def generate_report(question, rag_answer, comparison, hypothesis, experiment_plan):
    evidence_lines = []

    for item in retrieve(question, top_k=8):
        evidence_lines.append(
            f"- **{item['paper']}**, page {item['page']} "
            f"(similarity {item['score']:.3f})"
        )

    report = f'''# SciGenX Research Analysis Report

## Research Question
{question}

## Evidence-Based Answer
{rag_answer}

## Supporting Evidence
{chr(10).join(evidence_lines)}

## Paper Comparison
{comparison}

## Potential Research Gap
The gap below is a hypothesis generated from the analyzed literature and should be validated against additional and newer publications.

{comparison}

## Testable Hypothesis
{hypothesis}

## Experimental Plan
{experiment_plan}

## Limitations
- The analysis depends on the papers supplied to SciGenX.
- PDF extraction may miss tables, figures, equations, or scanned text.
- Retrieval quality depends on chunking and embedding quality.
- LLM-generated synthesis should be checked against the original papers.
- A potential research gap is not proof of novelty.

## Reproducibility
SciGenX MVP uses:
- PyMuPDF for PDF extraction
- Sentence Transformers for embeddings
- FAISS for vector retrieval
- Ollama with `llama3.2:latest` for local LLM generation
'''

    return report


research_question = "What are the main approaches, limitations, and potential research gaps in these papers?"

final_report = generate_report(
    research_question,
    rag_result["answer"],
    comparison,
    hypothesis,
    experiment_plan
)

print(final_report[:10000])


# SciGenX Research Analysis Report

## Research Question
What are the main approaches, limitations, and potential research gaps in these papers?

## Evidence-Based Answer
Based on the retrieved evidence, the main methods used in the papers are:

1. Convolutional Neural Networks (CNNs): 121 studies used CNNs for plant disease recognition, with a predominance of TensorFlow as the framework used (Evidence 5, Page 8).
2. Transfer Learning: 25 studies proposed new architectures, but 15 investigations customized classic algorithms to solve the problem of identifying plant diseases using CNNs (Evidence 5, Page 8).
3. Image Segmentation: 7 groups of methods were categorized, including Image Segmentation, as a practical approach (Evidence 3, Page 7).
4. Data Augmentation: 66 studies used customized data sets, with 45 studies using the PlantVillage data set (Evidence 3, Page 7).
5. Unsupervised Learning: 7 groups of methods were categorized, including Unsupervised Learning, as a practical approa

## Step 19 — Save the Markdown report

In [24]:
REPORT_PATH = REPORTS_DIR / "scigenx_research_report.md"

REPORT_PATH.write_text(final_report, encoding="utf-8")

print("Report saved to:", REPORT_PATH)


Report saved to: c:\Users\siris\OneDrive\Desktop\SciGenX\reports\scigenx_research_report.md


## Step 20 — Optional: Streamlit-ready data

The main research pipeline is now complete. A Streamlit dashboard can call the same functions:
- `retrieve()`
- `ask_rag()`
- `analyze_paper()`
- `compare_papers()`
- `generate_hypothesis()`
- `generate_experiment_plan()`

This keeps the notebook as the development/prototype environment and the Streamlit app as the user interface.
